# CNN-EXP-046 — Regional Activity Gate for Base-Level Ranking

**Oracle-only stage.** Primary region size is pre-registered at **32 bp**. This notebook does **not** train a regional head and must not use validation for training, hard-negative mining, calibration, alpha selection, or hyperparameter selection.

The notebook first reproduces the EXP045 natural full-validation nucleotide AP1, then computes a diagnostic label oracle for 16/32/64/128 bp fixed genomic regions. Region identity is strand-specific: `(contig, strand, position // R)`. Minus-track predictions are returned from reverse-complement order to genomic left→right before region assignment.

### Interpretation boundary

Oracle32 is an upper-bound diagnostic only for the mechanism **“perfectly suppress truly empty regions while preserving EXP045 base-level ordering among all retained positions.”** Oracle32 < 0.20 does **not** mathematically disprove every possible regional factorization, because a learned regional head could also reorder active regions. Under the current compute/resource policy, however, Oracle32 < 0.20 is a pre-registered **NO-GO for launching the frozen-head pilot**.

### Exact integrity invariants

Before any oracle result is accepted:

- evaluated position-strand count must equal **83,807,486**;
- genomic positive coordinate count must equal **56,205**;
- positive coordinates reconstructed through EXP045 minus-RC evaluation orientation must match the raw genomic-label coordinates **exactly as sets of integer tuples**;
- for every R, retained positive count must equal **56,205** exactly;
- dropped positive count must equal **0** exactly;
- fixed LOW_TP retained count must equal fixed LOW_TP original count exactly.

Any violation means coordinate/strand/crop/whitelist handling is invalid and the oracle must be rejected.


In [1]:
from pathlib import Path
import gc
import hashlib
import json
import math
import os
import sys
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch_directml

# ------------------------------------------------------------
# HARD ENVIRONMENT / PROJECT GUARDS
# ------------------------------------------------------------
EXPECTED_PYTHON = os.path.normcase(os.path.abspath(
    r"C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe"
))
ACTUAL_PYTHON = os.path.normcase(os.path.abspath(sys.executable))
assert ACTUAL_PYTHON == EXPECTED_PYTHON, (
    f"Wrong interpreter: {sys.executable}\nExpected: {EXPECTED_PYTHON}"
)

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate
    for candidate in (CURRENT_DIR, *CURRENT_DIR.parents)
    if (candidate / "README.md").is_file()
    and (candidate / "src" / "ml_project").is_dir()
)

EXPECTED_PROJECT_ROOT = Path(r"D:\Projects\EPIC\EPIC").resolve()
assert os.path.normcase(str(PROJECT_ROOT)) == os.path.normcase(str(EXPECTED_PROJECT_ROOT)), (
    f"Wrong project root: {PROJECT_ROOT}"
)

AGENTS_PATH = PROJECT_ROOT.parent / "AGENTS.md"
assert AGENTS_PATH.is_file(), f"Missing required {AGENTS_PATH}"
AGENTS_SHA256 = hashlib.sha256(AGENTS_PATH.read_bytes()).hexdigest()
print("Project:", PROJECT_ROOT)
print("Interpreter:", sys.executable)
print("AGENTS.md SHA256:", AGENTS_SHA256)

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ml_project.epic_baseline import load_baseline_sequences
from ml_project.epic_cnn import (
    ProfileWindowConfig,
    ProfileWindowGenerator,
    create_cnn_presence_intensity,
    forward_both_strands_multitask,
)
from ml_project.epic_data import load_split

EXPERIMENT_ID = "CNN-EXP-046"
EXPERIMENT_SLUG = "CNN_EXP046_REGIONAL_ACTIVITY_GATE"
RUN_DIR = PROJECT_ROOT / "artifacts" / "experiments" / EXPERIMENT_SLUG / "run_001"
RUN_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATHS = {
    "prereg": RUN_DIR / "EXP046_preregistration.json",
    "oracle_csv": RUN_DIR / "oracle_regional_gate_results.csv",
    "oracle_json": RUN_DIR / "oracle_regional_gate_summary.json",
    "lowtp_csv": RUN_DIR / "fixed_low_tp_coordinates.csv",
    "hardfp_csv": RUN_DIR / "fixed_hard_fp_coordinates.csv",
    "plot_ap": RUN_DIR / "oracle_ap_by_region_size.png",
    "plot_fp": RUN_DIR / "oracle_hard_fp_empty_fraction.png",
}
for name, path in OUTPUT_PATHS.items():
    assert not path.exists(), f"Refusing to overwrite existing output {name}: {path}"

REGION_SIZES = (16, 32, 64, 128)
PRIMARY_REGION_SIZE = 32
GRID_ORIGIN = 0
LOW_TP_Q25 = 0.7146850824356079  # frozen from EXP045
HARD_NEG_PER_TILE_STRAND = 2
HARD_FP_KEEP = 20_000
EVAL_BATCH_SIZE = 1
PROGRESS_EVERY = 250

EXPECTED_TOTAL_POSITION_STRAND = 83_807_486
EXPECTED_POSITIVES = 56_205
EXPECTED_AP = {
    "overall": 0.09763009437871803,
    "NC_064034.1": 0.09569052125003817,
    "NC_064041.1": 0.09140733554044979,
    "NC_064042.1": 0.10520206334231234,
}
BASELINE_AP_TOL = 5e-6

EXP045_RUN_DIR = (
    PROJECT_ROOT / "artifacts" / "experiments" /
    "CNN_EXP045_WIDTH512_50K_TWO_PHASE_LR_RF1029" / "run_001"
)
CHECKPOINT_PATH = EXP045_RUN_DIR / "width512_50k_final.pt"
assert CHECKPOINT_PATH.is_file(), f"Missing EXP045 checkpoint: {CHECKPOINT_PATH}"

preregistration = {
    "experiment_id": EXPERIMENT_ID,
    "reference": "CNN-EXP-045 WIDTH512 50k",
    "checkpoint": str(CHECKPOINT_PATH),
    "primary_region_size": PRIMARY_REGION_SIZE,
    "diagnostic_region_sizes": list(REGION_SIZES),
    "grid_origin": GRID_ORIGIN,
    "region_key": "(contig, strand, floor(absolute_position / R))",
    "oracle_only_uses_validation_labels_for_diagnostic_potential": True,
    "validation_for_training_or_mining": False,
    "go_threshold_oracle32_ap1": 0.20,
    "oracle_scope": "perfect empty-region suppression while preserving EXP045 ordering among retained positions",
    "no_go_scope": "resource-policy no-go for frozen-head pilot; not a proof against every regional factorization",
    "agents_sha256": AGENTS_SHA256,
}
OUTPUT_PATHS["prereg"].write_text(
    json.dumps(preregistration, indent=2, ensure_ascii=False), encoding="utf-8"
)
print("Run dir:", RUN_DIR)
print("Checkpoint:", CHECKPOINT_PATH)


Project: D:\Projects\EPIC\EPIC
Interpreter: C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe
AGENTS.md SHA256: bbc36fcf2270d4ce806e25a5be0d6b7eeab5fe80bc91acd040e71459baac68c3
Run dir: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP046_REGIONAL_ACTIVITY_GATE\run_001
Checkpoint: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP045_WIDTH512_50K_TWO_PHASE_LR_RF1029\run_001\width512_50k_final.pt


In [2]:
# ------------------------------------------------------------
# LOAD SPLIT / MODEL / VALIDATION GENERATOR
# ------------------------------------------------------------
split = load_split(PROJECT_ROOT)
sequences, fasta_alphabet = load_baseline_sequences(split)

train_contigs = set(split.intervals("train")["contig"].astype(str))
validation_contigs = set(split.intervals("validation")["contig"].astype(str))
assert train_contigs.isdisjoint(validation_contigs)
assert len(train_contigs) == 9
assert len(validation_contigs) == 3

profile_config = ProfileWindowConfig(
    target_length=1024,
    context_flank=130,
    positive_branch_fraction=0.5,
)

checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
model = create_cnn_presence_intensity(channels=512)
model.load_state_dict(checkpoint["model_state_dict"])
assert int(model.backbone.stem.out_channels) == 512
assert int(model.profile_head.in_channels) == 512
assert int(model.profile_head.out_channels) == 2
assert int(checkpoint["step"]) == 50_000

device = torch_directml.device()
model = model.to(device)
model.eval()

validation_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="validation",
    config=profile_config,
    seed=42,
)
validation_tiles = validation_generator.tile_index.copy().reset_index(drop=True)

required_columns = {"contig", "target_start", "target_end", "allowed_position_strand"}
missing_columns = required_columns - set(validation_tiles.columns)
assert not missing_columns, f"tile_index missing columns: {sorted(missing_columns)}"

assert int(validation_tiles["allowed_position_strand"].sum()) == EXPECTED_TOTAL_POSITION_STRAND
assert int(validation_generator.total_positive) == EXPECTED_POSITIVES
assert (validation_tiles["target_end"] - validation_tiles["target_start"] == profile_config.target_length).all()

# Target tiles must not overlap in genomic coordinates within a contig.
# Overlap would double-count positions and make absolute region assignment ambiguous.
for _contig, _frame in validation_tiles.groupby("contig", sort=True):
    _frame = _frame.sort_values("target_start")
    _starts = _frame["target_start"].to_numpy(dtype=np.int64)
    _ends = _frame["target_end"].to_numpy(dtype=np.int64)
    if len(_frame) > 1:
        assert np.all(_starts[1:] >= _ends[:-1]), f"Overlapping target tiles on {_contig}"

# Fast one-pass oracle is valid only if every requested region is fully contained in a tile.
# Because 128 is the largest R and all smaller R divide 128, 128-alignment is sufficient.
GRID_ALIGNED_TO_TILES = bool(
    (validation_tiles["target_start"].astype(np.int64) % max(REGION_SIZES) == 0).all()
    and (validation_tiles["target_end"].astype(np.int64) % max(REGION_SIZES) == 0).all()
)

print("DirectML:", torch_directml.device_name(0))
print("Validation tiles:", len(validation_tiles))
print("Validation contigs:", sorted(validation_contigs))
print("Grid aligned to 128-bp boundaries:", GRID_ALIGNED_TO_TILES)
print("Checkpoint model channels:", model.backbone.stem.out_channels)

del checkpoint
gc.collect()


DirectML: AMD Radeon RX 6750 GRE 12GB 
Validation tiles: 47743
Validation contigs: ['NC_064034.1', 'NC_064041.1', 'NC_064042.1']
Grid aligned to 128-bp boundaries: True
Checkpoint model channels: 512


30

In [3]:
# ------------------------------------------------------------
# METRIC HELPERS — SAME AP IMPLEMENTATION AS EXP045
# ------------------------------------------------------------
SUBMISSION_LEVELS = 100_001

def sigmoid_numpy(x):
    x = np.asarray(x, dtype=np.float64)
    out = np.empty_like(x, dtype=np.float64)
    positive = x >= 0
    out[positive] = 1.0 / (1.0 + np.exp(-x[positive]))
    exp_x = np.exp(x[~positive])
    out[~positive] = exp_x / (1.0 + exp_x)
    return out.astype(np.float32)

def probability_bin_5dp(probabilities):
    return np.rint(np.clip(probabilities, 0.0, 1.0) * 100_000).astype(np.int32)

def update_histogram(hist, indices):
    values, counts = np.unique(indices, return_counts=True)
    hist[values] += counts.astype(np.int64)

def ap_from_histograms(positive_hist, total_hist):
    positives = int(positive_hist.sum())
    if positives <= 0:
        return float("nan")
    pos_desc = positive_hist[::-1]
    total_desc = total_hist[::-1]
    tp = np.cumsum(pos_desc, dtype=np.int64)
    predicted_positive = np.cumsum(total_desc, dtype=np.int64)
    precision = tp / np.maximum(predicted_positive, 1)
    return float(np.sum((pos_desc / positives) * precision))

def new_hist_state():
    return {
        "total_hist": np.zeros(SUBMISSION_LEVELS, dtype=np.int64),
        "positive_hist": np.zeros(SUBMISSION_LEVELS, dtype=np.int64),
    }

def add_scores(state, scores, positive_mask):
    bins = probability_bin_5dp(scores)
    update_histogram(state["total_hist"], bins)
    if positive_mask.any():
        update_histogram(state["positive_hist"], bins[positive_mask])

def new_diag_state():
    return {
        "negative_n": 0,
        "negative_active_n": 0,
        "negative_empty_n": 0,
        "positive_n": 0,
        "positive_retained_n": 0,
        "low_tp_n": 0,
        "low_tp_retained_n": 0,
    }


In [4]:
# ------------------------------------------------------------
# OPTIONAL LABEL-ONLY PREPASS IF TILE GRID IS NOT REGION-ALIGNED
# ------------------------------------------------------------
active_region_sets = None

if not GRID_ALIGNED_TO_TILES:
    print("Tile boundaries are not 128-bp aligned; using a label-only prepass for exact global regions.")
    active_region_sets = {
        R: {(contig, strand): set() for contig in sorted(validation_contigs) for strand in ("+", "-")}
        for R in REGION_SIZES
    }
    for row_i in range(len(validation_tiles)):
        selected = validation_tiles.iloc[row_i:row_i + 1]
        row = selected.iloc[0]
        contig = str(row["contig"])
        target_start = int(row["target_start"])
        target_end = int(row["target_end"])
        assert target_end - target_start == profile_config.target_length
        abs_pos = target_start + np.arange(profile_config.target_length, dtype=np.int64)
        batch = validation_generator.make_batch(selected)
        for strand, count, mask in (
            ("+", batch.count[0, 0, :].numpy(), batch.mask[0, 0, :].numpy().astype(bool)),
            ("-", batch.count[0, 1, :].numpy(), batch.mask[0, 1, :].numpy().astype(bool)),
        ):
            positive_abs = abs_pos[(count > 0) & mask]
            for R in REGION_SIZES:
                ids = np.unique((positive_abs - GRID_ORIGIN) // R)
                active_region_sets[R][(contig, strand)].update(ids.tolist())
        del batch, selected
        if (row_i + 1) % 1000 == 0:
            print(f"label prepass {row_i + 1}/{len(validation_tiles)}")
    gc.collect()
else:
    print("One-pass exact oracle enabled: tile boundaries contain all requested regions.")


One-pass exact oracle enabled: tile boundaries contain all requested regions.


In [5]:
# ------------------------------------------------------------
# FULL NATURAL VALIDATION + ORACLE DIAGNOSTIC
# ------------------------------------------------------------
segment_names = ["overall", *sorted(validation_contigs)]
baseline_states = {segment: new_hist_state() for segment in segment_names}
oracle_states = {
    R: {segment: new_hist_state() for segment in segment_names}
    for R in REGION_SIZES
}
diag_states = {
    R: {segment: new_diag_state() for segment in segment_names}
    for R in REGION_SIZES
}

hard_score_parts = []
hard_position_parts = []
hard_contig_parts = []
hard_strand_parts = []
hard_active_parts = {R: [] for R in REGION_SIZES}

lowtp_score_parts = []
lowtp_count_parts = []
lowtp_position_parts = []
lowtp_contig_parts = []
lowtp_strand_parts = []
lowtp_active_parts = {R: [] for R in REGION_SIZES}

positions_seen = 0
positives_seen = 0

# Exact coordinate audit. Only positive coordinates are retained in memory (~56k tuples).
positive_coords_genomic = set()
positive_coords_exp045_roundtrip = set()

started = time.perf_counter()

with torch.no_grad():
    for batch_no, start in enumerate(range(0, len(validation_tiles), EVAL_BATCH_SIZE), 1):
        selected = validation_tiles.iloc[start:start + EVAL_BATCH_SIZE]
        batch = validation_generator.make_batch(selected)
        B = len(selected)
        x_gpu = batch.x_both_strands.to(device)

        plus_presence_gpu, minus_presence_reverse_gpu, _, _ = forward_both_strands_multitask(
            model,
            x_gpu,
            batch_size=B,
            context_flank=profile_config.context_flank,
            target_length=profile_config.target_length,
        )

        plus_logit = plus_presence_gpu[:, 0, :].detach().cpu().numpy().astype(np.float32, copy=True)
        minus_logit_reverse = minus_presence_reverse_gpu[:, 0, :].detach().cpu().numpy().astype(np.float32, copy=True)
        plus_probability = sigmoid_numpy(plus_logit)

        # Critical coordinate correction: model minus output is in RC order.
        # Flip predictions back to genomic left->right before assigning target_start + index.
        minus_probability = sigmoid_numpy(minus_logit_reverse)[:, ::-1].copy()

        del x_gpu, plus_presence_gpu, minus_presence_reverse_gpu

        plus_count = batch.count[:, 0, :].numpy().astype(np.float32, copy=False)
        minus_count = batch.count[:, 1, :].numpy().astype(np.float32, copy=False)
        plus_mask = batch.mask[:, 0, :].numpy().astype(bool, copy=False)
        minus_mask = batch.mask[:, 1, :].numpy().astype(bool, copy=False)

        for local_i in range(B):
            row = selected.iloc[local_i]
            contig = str(row["contig"])
            target_start = int(row["target_start"])
            target_end = int(row["target_end"])
            L = profile_config.target_length
            assert target_end - target_start == L
            abs_pos = target_start + np.arange(L, dtype=np.int64)

            # --------------------------------------------------------
            # EXACT STRAND/COORDINATE ROUND-TRIP AUDIT
            # --------------------------------------------------------
            # Raw batch labels are genomic left->right for both strands.
            raw_plus_positive_pos = abs_pos[(plus_count[local_i] > 0) & plus_mask[local_i]]
            raw_minus_positive_pos = abs_pos[(minus_count[local_i] > 0) & minus_mask[local_i]]

            positive_coords_genomic.update(
                (contig, "+", int(p)) for p in raw_plus_positive_pos
            )
            positive_coords_genomic.update(
                (contig, "-", int(p)) for p in raw_minus_positive_pos
            )

            # Reconstruct the exact coordinate convention used by EXP045:
            # minus labels/mask are flipped to RC order, where index j maps to
            # genomic position target_end - 1 - j. Mapping back must recover
            # the same integer genomic positive coordinates exactly.
            exp045_plus_pos = abs_pos[(plus_count[local_i] > 0) & plus_mask[local_i]]
            minus_count_reverse_audit = minus_count[local_i][::-1]
            minus_mask_reverse_audit = minus_mask[local_i][::-1]
            rc_abs_pos = (target_end - 1) - np.arange(L, dtype=np.int64)
            exp045_minus_pos = rc_abs_pos[
                (minus_count_reverse_audit > 0) & minus_mask_reverse_audit
            ]

            positive_coords_exp045_roundtrip.update(
                (contig, "+", int(p)) for p in exp045_plus_pos
            )
            positive_coords_exp045_roundtrip.update(
                (contig, "-", int(p)) for p in exp045_minus_pos
            )

            for strand, probability, count, mask in (
                ("+", plus_probability[local_i], plus_count[local_i], plus_mask[local_i]),
                ("-", minus_probability[local_i], minus_count[local_i], minus_mask[local_i]),
            ):
                valid_probability = probability[mask]
                valid_count = count[mask]
                valid_abs_pos = abs_pos[mask]
                positive = valid_count > 0
                negative = ~positive

                positions_seen += int(mask.sum())
                positives_seen += int(positive.sum())

                for segment in ("overall", contig):
                    add_scores(baseline_states[segment], valid_probability, positive)

                lowtp = positive & (valid_probability <= LOW_TP_Q25)
                if lowtp.any():
                    lowtp_score_parts.append(valid_probability[lowtp].astype(np.float32, copy=True))
                    lowtp_count_parts.append(valid_count[lowtp].astype(np.float32, copy=True))
                    lowtp_position_parts.append(valid_abs_pos[lowtp].astype(np.int64, copy=True))
                    lowtp_contig_parts.append(np.full(int(lowtp.sum()), contig, dtype=object))
                    lowtp_strand_parts.append(np.full(int(lowtp.sum()), strand, dtype="U1"))

                active_valid_by_R = {}
                for R in REGION_SIZES:
                    region_id_full = (abs_pos - GRID_ORIGIN) // R
                    if active_region_sets is None:
                        active_ids = np.unique(region_id_full[(count > 0) & mask])
                        active_full = np.isin(region_id_full, active_ids)
                    else:
                        active_ids = active_region_sets[R][(contig, strand)]
                        active_full = np.fromiter(
                            (int(x) in active_ids for x in region_id_full),
                            dtype=bool,
                            count=L,
                        )
                    active_valid = active_full[mask]
                    active_valid_by_R[R] = active_valid

                    # Label-oracle integrity: every positive must be in an active region.
                    assert bool(np.all(active_valid[positive]))
                    oracle_probability = np.where(active_valid, valid_probability, 0.0).astype(np.float32)

                    for segment in ("overall", contig):
                        add_scores(oracle_states[R][segment], oracle_probability, positive)
                        d = diag_states[R][segment]
                        d["negative_n"] += int(negative.sum())
                        d["negative_active_n"] += int((negative & active_valid).sum())
                        d["negative_empty_n"] += int((negative & ~active_valid).sum())
                        d["positive_n"] += int(positive.sum())
                        d["positive_retained_n"] += int((positive & active_valid).sum())
                        d["low_tp_n"] += int(lowtp.sum())
                        d["low_tp_retained_n"] += int((lowtp & active_valid).sum())

                    if lowtp.any():
                        lowtp_active_parts[R].append(active_valid[lowtp].astype(bool, copy=True))

                # Fixed EXP045-style HARD_FP candidate cohort: top 2 negatives per tile/strand.
                negative_index = np.flatnonzero(negative)
                if len(negative_index):
                    n_hard = min(HARD_NEG_PER_TILE_STRAND, len(negative_index))
                    neg_score = valid_probability[negative_index]
                    if len(negative_index) <= n_hard:
                        hard_local = negative_index
                    else:
                        top_local = np.argpartition(neg_score, -n_hard)[-n_hard:]
                        hard_local = negative_index[top_local]

                    hard_score_parts.append(valid_probability[hard_local].astype(np.float32, copy=True))
                    hard_position_parts.append(valid_abs_pos[hard_local].astype(np.int64, copy=True))
                    hard_contig_parts.append(np.full(len(hard_local), contig, dtype=object))
                    hard_strand_parts.append(np.full(len(hard_local), strand, dtype="U1"))
                    for R in REGION_SIZES:
                        hard_active_parts[R].append(active_valid_by_R[R][hard_local].astype(bool, copy=True))

        del (
            batch, selected, plus_logit, minus_logit_reverse,
            plus_probability, minus_probability,
            plus_count, minus_count, plus_mask, minus_mask,
        )
        if batch_no % 25 == 0:
            gc.collect()
        if batch_no == 1 or batch_no % PROGRESS_EVERY == 0 or start + B >= len(validation_tiles):
            elapsed = time.perf_counter() - started
            print(
                f"tiles={start + B:>6}/{len(validation_tiles)} "
                f"positions={positions_seen:>12,} "
                f"positives={positives_seen:>7,} "
                f"positions/s={positions_seen / elapsed:,.0f}"
            )

assert positions_seen == EXPECTED_TOTAL_POSITION_STRAND, (positions_seen, EXPECTED_TOTAL_POSITION_STRAND)
assert positives_seen == EXPECTED_POSITIVES, (positives_seen, EXPECTED_POSITIVES)

# Exact integer-coordinate invariants: no approximate float comparison here.
assert len(positive_coords_genomic) == EXPECTED_POSITIVES, (
    len(positive_coords_genomic), EXPECTED_POSITIVES
)
assert len(positive_coords_exp045_roundtrip) == EXPECTED_POSITIVES, (
    len(positive_coords_exp045_roundtrip), EXPECTED_POSITIVES
)
assert positive_coords_genomic == positive_coords_exp045_roundtrip, (
    "Genomic positive coordinates do not match EXP045 RC->genomic round-trip. "
    "Reject oracle: strand/coordinate/crop/whitelist bug."
)

print("Exact positive-coordinate round-trip: PASS")
print("Unique genomic positive coordinates:", len(positive_coords_genomic))
print("FULL ORACLE PASS FINISHED")


tiles=     1/47743 positions=         210 positives=      0 positions/s=1,482
tiles=   250/47743 positions=     326,316 positives=     11 positions/s=50,173
tiles=   500/47743 positions=     708,204 positives=    445 positions/s=54,716
tiles=   750/47743 positions=   1,102,632 positives=    790 positions/s=56,797
tiles=  1000/47743 positions=   1,519,462 positives=  1,034 positions/s=58,813
tiles=  1250/47743 positions=   1,928,284 positives=  1,209 positions/s=59,772
tiles=  1500/47743 positions=   2,381,142 positives=  1,626 positions/s=61,510
tiles=  1750/47743 positions=   2,825,034 positives=  1,980 positions/s=62,557
tiles=  2000/47743 positions=   3,270,676 positives=  2,259 positions/s=63,391
tiles=  2250/47743 positions=   3,688,504 positives=  2,496 positions/s=63,564
tiles=  2500/47743 positions=   4,124,072 positives=  2,890 positions/s=63,964
tiles=  2750/47743 positions=   4,591,424 positives=  3,262 positions/s=64,730
tiles=  3000/47743 positions=   5,044,192 positives= 

In [6]:
# ------------------------------------------------------------
# BASELINE REPRODUCTION GATE + ORACLE TABLE
# ------------------------------------------------------------
baseline_ap = {
    segment: ap_from_histograms(state["positive_hist"], state["total_hist"])
    for segment, state in baseline_states.items()
}

for segment, expected in EXPECTED_AP.items():
    actual = baseline_ap[segment]
    delta = actual - expected
    print(f"baseline {segment}: actual={actual:.12f} expected={expected:.12f} delta={delta:+.3e}")
    assert abs(delta) <= BASELINE_AP_TOL, (
        f"EXP045 reproduction failed for {segment}: {actual} vs {expected}"
    )

result_rows = []
for R in REGION_SIZES:
    for segment in segment_names:
        d = diag_states[R][segment]
        oracle_ap = ap_from_histograms(
            oracle_states[R][segment]["positive_hist"],
            oracle_states[R][segment]["total_hist"],
        )
        result_rows.append({
            "region_size": R,
            "primary_region": R == PRIMARY_REGION_SIZE,
            "segment": segment,
            "baseline_ap1": baseline_ap[segment],
            "oracle_ap1": oracle_ap,
            "delta_ap1": oracle_ap - baseline_ap[segment],
            "negative_n": d["negative_n"],
            "negative_active_fraction": d["negative_active_n"] / max(d["negative_n"], 1),
            "negative_empty_fraction": d["negative_empty_n"] / max(d["negative_n"], 1),
            "positive_recall": d["positive_retained_n"] / max(d["positive_n"], 1),
            "low_tp_retention": d["low_tp_retained_n"] / max(d["low_tp_n"], 1),
        })

results = pd.DataFrame(result_rows)

# Exact count invariants. Float ratios are report-only; they are not the integrity gate.
for R in REGION_SIZES:
    for segment in segment_names:
        d = diag_states[R][segment]
        assert d["positive_retained_n"] == d["positive_n"], (
            R, segment, d["positive_retained_n"], d["positive_n"]
        )
        assert d["positive_n"] - d["positive_retained_n"] == 0
        assert d["low_tp_retained_n"] == d["low_tp_n"], (
            R, segment, d["low_tp_retained_n"], d["low_tp_n"]
        )

    overall_d = diag_states[R]["overall"]
    assert overall_d["positive_n"] == EXPECTED_POSITIVES
    assert overall_d["positive_retained_n"] == EXPECTED_POSITIVES

print("Exact retained-positive / LOW_TP invariants: PASS")

# Fixed LOW_TP coordinate cohort.
lowtp_df = pd.DataFrame({
    "contig": np.concatenate(lowtp_contig_parts),
    "strand": np.concatenate(lowtp_strand_parts),
    "position": np.concatenate(lowtp_position_parts),
    "baseline_score": np.concatenate(lowtp_score_parts),
    "count": np.concatenate(lowtp_count_parts),
})
for R in REGION_SIZES:
    lowtp_df[f"active_r{R}"] = np.concatenate(lowtp_active_parts[R])
lowtp_df = lowtp_df.sort_values(["contig", "strand", "position"]).reset_index(drop=True)
assert len(lowtp_df) == 14_052, f"Unexpected fixed LOW_TP size: {len(lowtp_df)}"

# Fixed HARD_FP coordinates: reproduce EXP045 candidate protocol then retain global top 20k.
hard_all = pd.DataFrame({
    "contig": np.concatenate(hard_contig_parts),
    "strand": np.concatenate(hard_strand_parts),
    "position": np.concatenate(hard_position_parts),
    "baseline_score": np.concatenate(hard_score_parts),
})
for R in REGION_SIZES:
    hard_all[f"active_r{R}"] = np.concatenate(hard_active_parts[R])

hard_keep = min(HARD_FP_KEEP, len(hard_all))
hard_idx = np.argpartition(hard_all["baseline_score"].to_numpy(), -hard_keep)[-hard_keep:]
hard_fp_df = hard_all.iloc[hard_idx].copy()
hard_fp_df = hard_fp_df.sort_values("baseline_score", ascending=False).reset_index(drop=True)
assert len(hard_fp_df) == HARD_FP_KEEP

hard_fp_summary = []
for R in REGION_SIZES:
    active = hard_fp_df[f"active_r{R}"].to_numpy(dtype=bool)
    hard_fp_summary.append({
        "region_size": R,
        "hard_fp_n": len(active),
        "hard_fp_active_fraction": float(active.mean()),
        "hard_fp_empty_fraction": float((~active).mean()),
    })
hard_fp_summary = pd.DataFrame(hard_fp_summary)

results = results.merge(hard_fp_summary, on="region_size", how="left")
results.to_csv(OUTPUT_PATHS["oracle_csv"], index=False)
lowtp_df.to_csv(OUTPUT_PATHS["lowtp_csv"], index=False)
hard_fp_df.to_csv(OUTPUT_PATHS["hardfp_csv"], index=False)

display(results.sort_values(["region_size", "segment"]))
display(hard_fp_summary)
print("LOW_TP fixed cohort:", len(lowtp_df))
print("HARD_FP fixed cohort:", len(hard_fp_df))


baseline overall: actual=0.097630094379 expected=0.097630094379 delta=+0.000e+00
baseline NC_064034.1: actual=0.095690521250 expected=0.095690521250 delta=+0.000e+00
baseline NC_064041.1: actual=0.091407335540 expected=0.091407335540 delta=+0.000e+00
baseline NC_064042.1: actual=0.105202063342 expected=0.105202063342 delta=+0.000e+00
Exact retained-positive / LOW_TP invariants: PASS


,region_size,primary_region,segment,baseline_ap1,oracle_ap1,delta_ap1,negative_n,negative_active_fraction,negative_empty_fraction,positive_recall,low_tp_retention,hard_fp_n,hard_fp_active_fraction,hard_fp_empty_fraction
1,16,False,NC_064034.1,0.095691,0.288433,0.192743,33925585,0.005794,0.994206,1.0,1.0,20000,0.16545,0.83455
2,16,False,NC_064041.1,0.091407,0.289986,0.198579,24132770,0.004903,0.995097,1.0,1.0,20000,0.16545,0.83455
3,16,False,NC_064042.1,0.105202,0.291227,0.186024,25692926,0.005963,0.994037,1.0,1.0,20000,0.16545,0.83455
0,16,False,overall,0.097630,0.289735,0.192105,83751281,0.005589,0.994411,1.0,1.0,20000,0.16545,0.83455
5,32,True,NC_064034.1,0.095691,0.236172,0.140482,33925585,0.010518,0.989482,1.0,1.0,20000,0.20790,0.79210
6,32,True,NC_064041.1,0.091407,0.237560,0.146152,24132770,0.008842,0.991158,1.0,1.0,20000,0.20790,0.79210
7,32,True,NC_064042.1,0.105202,0.241539,0.136336,25692926,0.010728,0.989272,1.0,1.0,20000,0.20790,0.79210
4,32,True,overall,0.097630,0.238318,0.140688,83751281,0.010100,0.989900,1.0,1.0,20000,0.20790,0.79210
9,64,False,NC_064034.1,0.095691,0.202981,0.107291,33925585,0.018862,0.981138,1.0,1.0,20000,0.24570,0.75430
10,64,False,NC_064041.1,0.091407,0.202723,0.111316,24132770,0.015874,0.984126,1.0,1.0,20000,0.24570,0.75430


,region_size,hard_fp_n,hard_fp_active_fraction,hard_fp_empty_fraction
0,16,20000,0.16545,0.83455
1,32,20000,0.20790,0.79210
2,64,20000,0.24570,0.75430
3,128,20000,0.28355,0.71645


LOW_TP fixed cohort: 14052
HARD_FP fixed cohort: 20000


In [7]:
# ------------------------------------------------------------
# DECISION + PLOTS + JSON SUMMARY
# ------------------------------------------------------------
overall = results[results["segment"] == "overall"].sort_values("region_size").copy()
primary_row = overall[overall["region_size"] == PRIMARY_REGION_SIZE].iloc[0]
primary_oracle_ap = float(primary_row["oracle_ap1"])

decision = "GO_FROZEN_PILOT" if primary_oracle_ap >= 0.20 else "NO_GO_REGIONAL_GATE_INSUFFICIENT"

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(overall["region_size"], overall["oracle_ap1"], marker="o", label="oracle AP1")
ax.axhline(0.20, linestyle="--", label="target 0.20")
ax.axhline(EXPECTED_AP["overall"], linestyle=":", label="EXP045 baseline")
ax.set_xlabel("Region size (bp)")
ax.set_ylabel("Natural nucleotide AP1")
ax.set_title("EXP046 oracle: regional activity gate")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_PATHS["plot_ap"], dpi=160)
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(
    hard_fp_summary["region_size"],
    hard_fp_summary["hard_fp_empty_fraction"],
    marker="o",
)
ax.set_xlabel("Region size (bp)")
ax.set_ylabel("Fraction of fixed HARD_FP in oracle-empty regions")
ax.set_ylim(0.0, 1.0)
ax.set_title("EXP046: suppressible fraction of fixed EXP045 HARD_FP")
fig.tight_layout()
fig.savefig(OUTPUT_PATHS["plot_fp"], dpi=160)
plt.close(fig)

summary = {
    "experiment_id": EXPERIMENT_ID,
    "diagnostic": "label oracle only; not a model result",
    "primary_region_size": PRIMARY_REGION_SIZE,
    "primary_oracle_ap1": primary_oracle_ap,
    "baseline_ap1": float(EXPECTED_AP["overall"]),
    "decision": decision,
    "criterion": "oracle32 AP1 >= 0.20 required for frozen pilot under current resource policy",
    "oracle_scope": "perfectly suppress empty 32-bp regions while preserving EXP045 ordering among retained positions",
    "scientific_caveat": "oracle32 < 0.20 does not disprove every regional factorization because a learned regional score can reorder active regions",
    "positive_recall_primary": float(primary_row["positive_recall"]),
    "low_tp_retention_primary": float(primary_row["low_tp_retention"]),
    "hard_fp_empty_fraction_primary": float(primary_row["hard_fp_empty_fraction"]),
    "grid_origin": GRID_ORIGIN,
    "region_is_strand_specific": True,
    "minus_prediction_flipped_to_genomic_orientation_before_region_assignment": True,
    "agents_sha256": AGENTS_SHA256,
    "outputs": {name: str(path) for name, path in OUTPUT_PATHS.items()},
}
OUTPUT_PATHS["oracle_json"].write_text(
    json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8"
)

print(json.dumps(summary, indent=2, ensure_ascii=False))
print()
if decision == "NO_GO_REGIONAL_GATE_INSUFFICIENT":
    print(
        "NO-GO under the current resource policy: perfect suppression of truly empty "
        "32-bp regions, while preserving EXP045 ordering among retained positions, does not "
        "reach AP1=0.20. This does not mathematically disprove every regional factorization, "
        "but it is insufficient evidence to justify the frozen-head pilot. Do not launch it."
    )
else:
    print(
        "GO: the 32-bp oracle clears AP1=0.20. "
        "A limited frozen-backbone regional-head pilot is justified; validation remains untouched for training/mining/tuning."
    )


{
  "experiment_id": "CNN-EXP-046",
  "diagnostic": "label oracle only; not a model result",
  "primary_region_size": 32,
  "primary_oracle_ap1": 0.23831825797792802,
  "baseline_ap1": 0.09763009437871803,
  "decision": "GO_FROZEN_PILOT",
  "criterion": "oracle32 AP1 >= 0.20 required for frozen pilot under current resource policy",
  "oracle_scope": "perfectly suppress empty 32-bp regions while preserving EXP045 ordering among retained positions",
  "scientific_caveat": "oracle32 < 0.20 does not disprove every regional factorization because a learned regional score can reorder active regions",
  "positive_recall_primary": 1.0,
  "low_tp_retention_primary": 1.0,
  "hard_fp_empty_fraction_primary": 0.7921,
  "grid_origin": 0,
  "region_is_strand_specific": true,
  "minus_prediction_flipped_to_genomic_orientation_before_region_assignment": true,
  "agents_sha256": "bbc36fcf2270d4ce806e25a5be0d6b7eeab5fe80bc91acd040e71459baac68c3",
  "outputs": {
    "prereg": "D:\\Projects\\EPIC\\EPIC\\ar

## Stop condition / next action

- If the primary 32-bp oracle is below 0.20: stop EXP046 before training the regional head.
- If it is at least 0.20 and every integrity assertion passed: prepare the separate frozen-backbone pilot notebook.
- Do not update the experiment registry/card until the diagnostic outputs have been reviewed.
